# 07 — Cross-Category Affinity Analysis

**Purpose:** Discover which categories are commonly shopped together. Enables cross-sell strategy.  
**Key question:** If a customer shops in category A, how much more likely are they to also shop in category B?  

**Tables used:** `marts.mart_category_affinity`

In [ ]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import os
PROJECT = os.environ.get('BQ_PROJECT', 'fmn-sandbox')
LOCATION = 'africa-south1'
client = bigquery.Client(project=PROJECT, location=LOCATION)

def q(sql):
    return client.query(sql).to_dataframe()

print(f'Connected to {PROJECT} ({LOCATION})')

## 1. Strongest category pairs by lift
Lift > 1.0 means they co-occur more than random chance. Lift of 2.0 = twice as likely.

In [ ]:
affinity = q(f"""
    SELECT category_a, category_b, shared_customers,
           customers_in_a, customers_in_b,
           pct_a_also_shops_b, pct_b_also_shops_a,
           lift, jaccard_pct
    FROM `{PROJECT}.marts.mart_category_affinity`
    WHERE lift > 1.0
    ORDER BY lift DESC
    LIMIT 30
""")

if not affinity.empty:
    print(f'Top 30 category pairs by lift:\n')
    affinity
else:
    print('No affinity data. Run mart_category_affinity SQL first.')

## 2. Affinity heatmap
Shows the lift between the top categories. Darker = stronger relationship.

In [ ]:
if not affinity.empty:
    # get top 15 categories by total shared customers
    all_cats = pd.concat([affinity['category_a'], affinity['category_b']]).value_counts().head(15).index.tolist()
    filtered = affinity[(affinity['category_a'].isin(all_cats)) & (affinity['category_b'].isin(all_cats))]

    # build symmetric matrix
    pivot = filtered.pivot(index='category_a', columns='category_b', values='lift').fillna(0)
    # make it symmetric
    for cat in all_cats:
        if cat not in pivot.index:
            pivot.loc[cat] = 0
        if cat not in pivot.columns:
            pivot[cat] = 0
    combined = pivot.add(pivot.T, fill_value=0)
    combined = combined.loc[sorted(combined.index), sorted(combined.columns)]

    fig = px.imshow(combined, text_auto='.1f', aspect='auto',
                    color_continuous_scale='YlOrRd',
                    title='Category affinity heatmap (lift)',
                    labels=dict(color='Lift'))
    fig.update_layout(height=700)
    fig.show()

## 3. Cross-sell opportunities
Categories with high lift but low overlap = untapped cross-sell potential.

In [ ]:
if not affinity.empty:
    # high lift but low jaccard = strong affinity but many customers havent crossed over yet
    cross_sell = affinity[(affinity['lift'] > 1.2) & (affinity['jaccard_pct'] < 30)].copy()
    cross_sell = cross_sell.sort_values('shared_customers', ascending=False).head(15)

    if not cross_sell.empty:
        print('Cross-sell opportunities (high affinity, low overlap):\n')
        for _, row in cross_sell.iterrows():
            print(f'  {row["category_a"]} + {row["category_b"]}: '
                  f'lift {row["lift"]:.1f}x, '
                  f'{row["shared_customers"]:,} shared, '
                  f'{row["pct_a_also_shops_b"]:.0f}% of A also in B')
    else:
        print('No strong cross-sell pairs found.')

## 4. Biggest overlaps by volume
Which category pairs share the most customers?

In [ ]:
biggest = q(f"""
    SELECT category_a, category_b, shared_customers, lift,
           pct_a_also_shops_b, pct_b_also_shops_a
    FROM `{PROJECT}.marts.mart_category_affinity`
    ORDER BY shared_customers DESC
    LIMIT 15
""")

if not biggest.empty:
    fig = px.bar(biggest, x='shared_customers',
                 y=biggest['category_a'] + ' + ' + biggest['category_b'],
                 orientation='h', color='lift',
                 color_continuous_scale='Viridis',
                 title='Largest category overlaps by shared customers',
                 labels={'y': 'Category pair', 'shared_customers': 'Shared customers'})
    fig.update_layout(yaxis=dict(autorange='reversed'), height=500)
    fig.show()